# Demo - Medallion layers and incremental processing

**Why we are doing this.** Kenny's complaint: they process entire datasets repeatedly instead of only new or changed records.

**What we will prove.**
1. A Bronze to Silver to Gold pipeline can be written as plain SELECT statements with a freshness target. Snowflake keeps them up to date (dynamic tables).
2. When 1,000 new rows arrive, only those 1,000 rows are processed, not millions.
3. The new column added during ingestion (DIALYSIS_MODALITY from the AETNA file) can be surfaced downstream with a one-line change.

**Before you start:** 00_Ingestion must have been run, so Bronze has data.

**For the Databricks engineers:** dynamic tables are closest to Delta Live Tables (declarative, incremental).

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **UI STEP:** something you do in the Snowsight screens or the AWS console, not in the notebook.
- **SAY THIS:** your talk track for that moment.

### SETUP - ALREADY DONE (read only, do not run)
**Silver: clean and de-duplicate members (latest record per member)**

```sql
CREATE DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
  TARGET_LAG = '1 minute'          -- how fresh the data must be
  WAREHOUSE = WH_TRANSFORM
  REFRESH_MODE = INCREMENTAL       -- process only changed rows
AS
SELECT MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
WHERE MEMBER_ID IS NOT NULL AND DOB IS NOT NULL AND LENGTH(STATE) = 2   -- business rules
QUALIFY ROW_NUMBER() OVER (PARTITION BY MEMBER_ID ORDER BY LOADED_AT DESC) = 1;  -- de-duplicate
```

### SETUP - ALREADY DONE (read only, do not run)
**Silver: flatten the nested lab JSON into one row per test**

```sql
CREATE DYNAMIC TABLE SOMATUS_DEMO.SILVER.LAB_RESULTS TARGET_LAG='1 minute' WAREHOUSE=WH_TRANSFORM REFRESH_MODE=INCREMENTAL AS
SELECT RAW:lab_id::STRING AS LAB_ID, RAW:member_id::STRING AS MEMBER_ID, RAW:client_id::STRING AS CLIENT_ID,
       RAW:collected_at::TIMESTAMP_NTZ AS COLLECTED_AT, r.value:test::STRING AS TEST,
       r.value:value::FLOAT AS RESULT_VALUE, r.value:unit::STRING AS UNIT, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.LABS_RAW, LATERAL FLATTEN(INPUT => RAW:panel.results) r;
```

### SETUP - ALREADY DONE (read only, do not run)
**Gold: members by client and kidney disease stage**

```sql
CREATE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY
  TARGET_LAG = '1 minute'          -- Gold stays within one minute of Silver
  WAREHOUSE = WH_TRANSFORM REFRESH_MODE = INCREMENTAL
AS SELECT CLIENT_ID, CKD_STAGE, COUNT(*) AS MEMBERS FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN GROUP BY 1, 2;
```

> **SAY THIS:** There is no orchestration code here. You write the SELECT and say how fresh it must be. Snowflake works out the dependencies and refreshes only what changed.

### RUN DURING DEMO
Set the role, warehouse and database. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_TRANSFORM;

### RUN DURING DEMO
List the dynamic tables. Point out the target_lag and refresh_mode columns.

In [ ]:
%%sql -r dts
SHOW DYNAMIC TABLES IN DATABASE SOMATUS_DEMO;

### UI STEP - do this in the screen, not in the notebook
**Show the pipeline graph**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO > GOLD > Dynamic Tables** and click **CLIENT_CKD_SUMMARY**.
3. Click the **Graph** tab. You see Bronze MEMBERS_RAW feeding Silver MEMBERS_CLEAN feeding Gold CLIENT_CKD_SUMMARY.
4. Click the **Refresh History** tab to show each refresh, how long it took and how many rows it changed.

## Step 1 - 1,000 new rows arrive: only those are processed

### RUN DURING DEMO
Insert 1,000 new members into Bronze. This stands in for a small new file arriving.

In [ ]:
%%sql -r insert_new
INSERT INTO SOMATUS_DEMO.BRONZE.MEMBERS_RAW
  (MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, FILE_NAME, LOADED_AT)
SELECT 'M8' || LPAD(SEQ4()::STRING, 7, '0'), 'HUMANA_DEMO', '1960-01-01', 'F', 'CKD5', 'VA',
       'live_demo_insert', CURRENT_TIMESTAMP()
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

### RUN DURING DEMO
Refresh now instead of waiting the one minute target lag, so the audience does not wait.

In [ ]:
%%sql -r refresh_now
ALTER DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN REFRESH;
ALTER DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY REFRESH;

### RUN DURING DEMO
The proof. REFRESH_ACTION is **INCREMENTAL** and about 1,000 rows were inserted, not 5 million. A refresh that shows NO_DATA means nothing changed, so nothing was processed and nothing was billed.

In [ ]:
%%sql -r refresh_proof
SELECT NAME, REFRESH_ACTION, STATE,
       STATISTICS:numInsertedRows::INT AS ROWS_INSERTED,
       STATISTICS:numDeletedRows::INT  AS ROWS_DELETED,
       DATEDIFF('millisecond', REFRESH_START_TIME, REFRESH_END_TIME) AS MILLISECONDS
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE NAME IN ('MEMBERS_CLEAN', 'CLIENT_CKD_SUMMARY')
ORDER BY REFRESH_START_TIME DESC LIMIT 6;

> **SAY THIS:** This is the end of reloading whole datasets. The engine only touches what changed since the last refresh.

## Step 2 - Surface the new column from ingestion downstream

When the AETNA file added DIALYSIS_MODALITY to Bronze, the Silver and Gold tables kept running untouched, because they list their columns explicitly. Showing the new column downstream is a deliberate, one-line change: add it to the SELECT. There is no reload and no pipeline rewrite.

### RUN DURING DEMO
Add DIALYSIS_MODALITY to Silver.

In [ ]:
%%sql -r evolve_silver
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
  TARGET_LAG = '1 minute' WAREHOUSE = WH_TRANSFORM REFRESH_MODE = INCREMENTAL AS
SELECT MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, DIALYSIS_MODALITY, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
WHERE MEMBER_ID IS NOT NULL AND DOB IS NOT NULL AND LENGTH(STATE) = 2
QUALIFY ROW_NUMBER() OVER (PARTITION BY MEMBER_ID ORDER BY LOADED_AT DESC) = 1;

### RUN DURING DEMO
Add a dialysis count to Gold.

In [ ]:
%%sql -r evolve_gold
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY
  TARGET_LAG = '1 minute' WAREHOUSE = WH_TRANSFORM REFRESH_MODE = INCREMENTAL AS
SELECT CLIENT_ID, CKD_STAGE, COUNT(*) AS MEMBERS, COUNT_IF(DIALYSIS_MODALITY IS NOT NULL) AS ON_DIALYSIS
FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN GROUP BY 1, 2;

### RUN DURING DEMO
Show the business-ready Gold table, now including the new dialysis information.

In [ ]:
%%sql -r gold_result
SELECT * FROM SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY ORDER BY CLIENT_ID, CKD_STAGE;